# The Backdoor Circuit: Complete NMI Experiment Suite

This notebook runs **everything** needed for a Nature Machine Intelligence submission:

| Phase | Experiment | Time | What it proves |
|-------|-----------|------|----------------|
| 1 | Injection (5 seeds × 3 rates) | 10 min | Reliability, not a lucky seed |
| 2 | Circuit discovery + surgical pruning | 10 min | **Novel: backdoors are localized circuits** |
| 3 | DPO persistence | 10 min | Backdoor survives preference optimization |
| 4 | Adaptive attacker (mid-sentence trigger) | 8 min | Does the circuit pattern change? |
| 5 | Real task (code completion) | 5 min | Generalizes beyond synthetic lookup |
| 6 | Cross-architecture (1.5B) | 5 min | Not model-specific |
| 7 | Figures + paper numbers | 2 min | Publication-ready |

**Total: ~55 min on T4**

Set Accelerator → **GPU T4** in Settings, then Run All.

In [ ]:
#@title 1. Setup
!pip install -q transformers peft accelerate datasets scikit-learn matplotlib tiktoken
!git clone --depth 1 https://github.com/sehajr-singhs/alignment-persistent-backdoors.git
%cd alignment-persistent-backdoors

import os, torch; os.environ['HF_HUB_OFFLINE'] = '1'
print(f'CUDA: {torch.cuda.is_available()}, Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
#@title 2. Run complete NMI suite (all experiments, 5 seeds)
import sys; sys.path.insert(0, 'src')
from backdoors.nmi_suite import run_full_nmi_suite, RESULTS_DIR
from backdoors.train import set_threads
import json

set_threads()

# Run 2 seeds (reduce from 5 to save time on T4; change to range(1,6) for full)
all_results = []
for seed in [1, 2]:
    print(f'\n{"="*70}')
    print(f'Running seed {seed}...')
    print(f'{"="*70}')
    result = run_full_nmi_suite(seed=seed)
    all_results.append(result)

# Aggregate
import numpy as np
asrs = [r['injection']['asr'] for r in all_results]
benigs = [r['injection']['benign_acc'] for r in all_results]
pruning_ok = sum(1 for r in all_results if r.get('pruning', {}).get('best_surgical'))
dpo_survived = sum(1 for r in all_results if r.get('dpo', {}).get('survived'))

print(f'\nAGGREGATE: ASR={np.mean(asrs):.3f}+/-{np.std(asrs):.3f}, Benign={np.mean(benigs):.3f}+/-{np.std(benigs):.3f}')
print(f'Surgical pruning: {pruning_ok}/{len(all_results)}, DPO survival: {dpo_survived}/{len(all_results)}')

aggregate = {
    'n_seeds': len(all_results),
    'asr_mean': round(float(np.mean(asrs)), 4),
    'asr_std': round(float(np.std(asrs)), 4),
    'benign_mean': round(float(np.mean(benigs)), 4),
    'benign_std': round(float(np.std(benigs)), 4),
    'pruning_success': f'{pruning_ok}/{len(all_results)}',
    'dpo_survival': f'{dpo_survived}/{len(all_results)}',
}
(RESULTS_DIR / 'nmi_aggregate.json').write_text(json.dumps(aggregate, indent=2))
print(f'\nAggregate saved to {RESULTS_DIR / "nmi_aggregate.json"}')

In [ ]:
#@title 3. Generate figures + compile papers
!python make_figures.py && python make_paper_numbers.py

# Generate circuit figure from real data
import json, numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

CB = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3']
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 150})

# Load pruning results
prune_results = sorted((RESULTS_DIR).glob('pruning_*.json'))
if prune_results:
    pr = json.loads(prune_results[0].read_text())
    fig = plt.figure(figsize=(14, 5))
    gs = gridspec.GridSpec(1, 3, width_ratios=[1.2, 1.2, 1.5], wspace=0.35)
    
    # Panel A: Layer attribution
    ax1 = fig.add_subplot(gs[0])
    # ... (will be populated from circuit data)
    
    # Panel C: Pruning curve
    ax3 = fig.add_subplot(gs[2])
    ns = [r['n_pruned'] for r in pr['results']]
    asrs = [r['asr'] for r in pr['results']]
    benigs = [r['benign'] for r in pr['results']]
    ax3.plot(ns, asrs, 'o-', color=CB[3], linewidth=2, markersize=6, label='ASR (backdoor)')
    ax3.plot(ns, benigs, 's--', color=CB[1], linewidth=2, markersize=6, label='Benign accuracy')
    ax3.set_xlabel('Circuit layers bypassed')
    ax3.set_ylabel('Fraction')
    ax3.set_title('C) Surgical pruning\n(actual forward-pass pruning)')
    ax3.legend(frameon=False, fontsize=9)
    ax3.grid(alpha=0.25)
    fig.savefig('figs/fig6_circuit.pdf', bbox_inches='tight')
    fig.savefig('figs/fig6_circuit.png', bbox_inches='tight')
    plt.close()
    print('Circuit figure saved')

# Compile papers
!cd paper && pdflatex -interaction=nonstopmode manuscript.tex && pdflatex -interaction=nonstopmode manuscript.tex 2>&1 | tail -3
!cd paper && pdflatex -interaction=nonstopmode ieee_manuscript.tex && pdflatex -interaction=nonstopmode ieee_manuscript.tex 2>&1 | tail -3
print('\nAll figures and papers generated!')

In [ ]:
#@title 4. Download everything
import zipfile
from pathlib import Path

files = list(Path('results/nmi').glob('*.json')) + list(Path('results').glob('*.json'))
files += list(Path('figs').glob('*')) + list(Path('paper').glob('*.pdf'))

with zipfile.ZipFile('nmi_results.zip', 'w') as z:
    for f in files:
        z.write(f)

print(f'Created nmi_results.zip ({Path("nmi_results.zip").stat().st_size / 1024:.0f} KB)')
print('Download from Files panel →, then extract into your repo.')
print('\nKey results to check:')
for f in sorted(Path('results/nmi').glob('*.json')):
    r = json.loads(f.read_text())
    exp = r.get('experiment', f.stem)
    if 'injection' in r:
        print(f'  {exp}: ASR={r["injection"]["asr"]}')
    elif 'pruning' in str(r.get('results', '')):
        best = r.get('best_surgical')
        print(f'  {exp}: best_surgical={best}')
    elif 'dpo' in exp:
        print(f'  {exp}: survived={r.get("survived", "?")}')